# CEBRA vs. Transformer: neural embeddings & position decoding

Benchmark **CEBRA** (contrastive neural-embedding learning; Schneider et al. 2023)
against the supervised **Transformer decoder** from `decode-position-comparison.ipynb`,
on the same `bat_data_loader` train/valid/test split (here: `DEFAULT_SESSION_NAMES`, the
18 canonical Dataset 1–3 "lab" sessions — Krishna sessions excluded).

CEBRA learns a low-D embedding of population spiking shaped to be consistent with
auxiliary behavior; a simple read-out then maps embedding → position. We evaluate:

- **CEBRA variants**
  - `cebra_pos` — behavior label = `(x, y, z)`; direction is *inferred* from the
    10-bin (1.0 s) `offset10-model` receptive field + `time_offsets=10`. Direct analog
    of the position-only Transformer input.
  - `cebra_posdir` — label = `(x, y, z, speed, v̂x, v̂y, v̂z)`; direction given
    **explicitly**, built exactly as the digitalbat `use_speed_head_dir` block.
  - `cebra_time` — `conditional='time'`, no label (self-supervised baseline).
- **Metric A** — continuous **linear** decode (Ridge) embedding → position, median 2D/3D
  error in meters + discrete-Fréchet trajectory distance, vs. the Transformer.
- **Metric B** — CEBRA `consistency_score` between runs (embedding reproducibility).
- **Metric C** — `consistency_score` across sessions/bats (aligned on per-arena-normalized
  position), grouped same-bat vs. cross-bat.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# CEBRA is source-only (not pip-installed); put it on the path first.
sys.path.insert(0, os.path.expanduser('~/Documents/code/CEBRA'))
sys.path.append(os.path.expanduser('~/Documents/code/utils'))
sys.path.append(os.path.expanduser('~/Documents/code/digitalbat'))

import cebra
from cebra import CEBRA
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import make_data_config, DEFAULT_SESSION_NAMES
from digitalbat.decoding import discrete_frechet
from digitalbat.models.decoder import Bat_Transformer_Decoder, bat_decoder_trainer
from digitalbat.utility.utils import set_random_state, bat_palette, parse_bat

print('cebra', cebra.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())

seed = 1
set_random_state(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BAT_PALETTE = bat_palette()

# CEBRA training length: 5000 for the real run (demo default); drop to e.g. 500
# for a quick smoke pass.
CEBRA_MAX_ITER = 5000
N_SEEDS = 3          # for the between-runs consistency metric
OFFSET = 10          # offset10-model receptive field (bins)
KNN_N = 15           # n_neighbors for the CEBRA-tutorial KNeighborsRegressor read-out
OUT_DIR = './out/cebra'
os.makedirs(OUT_DIR, exist_ok=True)
device

## 1. Dataloader

Same config as `decode-position-comparison.ipynb` / `preprocess-data-v0j-share.ipynb`,
but restricted to `DEFAULT_SESSION_NAMES`, so the split lines up. With all extra flags
off, `behavior_dim == 3` (pure `x, y, z`), `bin_width = 0.10 s`, flights only.

In [ ]:
data_config = make_data_config(session_names=DEFAULT_SESSION_NAMES)
session_names = data_config['session_names']
bin_width = data_config['bin_width']

dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)
assert behavior_dim == 3, f'expected pure xyz, got behavior_dim={behavior_dim}'
print(f'behavior_dim={behavior_dim}, sessions={len(unit_num_dict)}, bin_width={bin_width}s')
unit_num_dict

## 2. Per-flight tensors and helpers

CEBRA's `offset` models have a temporal receptive field, so we fit/transform **per
flight** (never across flight boundaries). Positions are returned in **meters**
(un-z-scored via `meta_info_dict['behavior_mean'/'behavior_std']`, exactly as the decode
notebook). Flights shorter than the receptive field are zero-padded then trimmed back.

`direction_label` reproduces the digitalbat `use_speed_head_dir` construction
(`vel = np.gradient(traj, bin_width); speed = |vel|; v̂ = vel/(speed+eps)`).

In [ ]:
def session_flights(session, split):
    """List of dicts per valid flight: spike (T,K) float32, pos_m (T,3) meters,
    flight_id, plus per-timestep t_idx. Mirrors decode-notebook un-z-scoring."""
    bm = np.asarray(meta_info_dict['behavior_mean'][session]).squeeze()[:3]
    bs = np.asarray(meta_info_dict['behavior_std'][session]).squeeze()[:3]
    out = []
    for batch in dataloaders[split][session]:
        m = batch.mask.numpy().astype(bool)        # (B, T)
        spike = batch.spike.numpy()                # (B, T, K)
        pos_z = batch.behavior[:, :, :3].numpy()   # (B, T, 3) z-scored
        fpi = batch.flight_path_index.numpy()      # (B,)
        for b in range(m.shape[0]):
            sel = m[b]
            if sel.sum() == 0:
                continue
            X = spike[b, sel].astype(np.float32)
            pos = (pos_z[b, sel] * bs[None] + bm[None]).astype(np.float32)
            t_idx = np.flatnonzero(sel).astype(int)   # within-flight timestep
            out.append(dict(spike=X, pos=pos, flight_id=int(fpi[b]), t_idx=t_idx))
    return out


def direction_label(pos):
    """(T,3) position (m) -> (T,7) [x,y,z,speed,v̂x,v̂y,v̂z]; digitalbat convention."""
    vel = np.gradient(pos, bin_width, axis=0)
    speed = np.linalg.norm(vel, axis=1)
    vhat = vel / (speed[:, None] + 1e-8)
    return np.concatenate([pos, speed[:, None], vhat], axis=1).astype(np.float32)


def stack_labels(flights, variant):
    """vstack the CEBRA label for a list of flights, per variant."""
    if variant == 'cebra_posdir':
        return np.vstack([direction_label(f['pos']) for f in flights])
    return np.vstack([f['pos'] for f in flights])   # cebra_pos / cebra_time use xyz (time uses none)


def _pad(X, n):
    if len(X) >= n:
        return X, len(X)
    pad = np.zeros((n - len(X), X.shape[1]), dtype=X.dtype)
    return np.vstack([X, pad]), len(X)


def embed_flights(model, flights):
    """transform each flight (padding short ones to the model's receptive field);
    return list of (emb (T,d), flight dict)."""
    req = len(getattr(model, 'offset_', range(OFFSET)))   # receptive field of this model
    out = []
    for f in flights:
        Xp, n = _pad(f['spike'], req)
        emb = model.transform(Xp)[:n]    # pad_before_transform=True keeps len == len(Xp)
        out.append((emb, f))
    return out


VARIANTS = ['cebra_pos', 'cebra_posdir', 'cebra_time']

# Fixed (non-searched) CEBRA settings — tutorial defaults held constant.
COMMON_FIXED = dict(batch_size=512, output_dimension=3, distance='cosine',
                    device='cuda_if_available', verbose=False)
# Searched HPs, centered on the rat-hippocampus tutorial defaults.  The Optuna
# search in Section 3 may overwrite these IN PLACE; until then they ARE the defaults.
CEBRA_HP = dict(model_architecture='offset10-model', learning_rate=3e-4,
                temperature=1.0, time_offsets=10)


def make_cebra(variant, max_iter):
    cond = 'time' if variant == 'cebra_time' else 'time_delta'
    return CEBRA(conditional=cond, max_iterations=max_iter, **COMMON_FIXED, **CEBRA_HP)


# Quick check on the first session.
_f = session_flights(session_names[0], 'train')
print(f'{session_names[0]}: {len(_f)} train flights, '
      f'label dims pos={stack_labels(_f, "cebra_pos").shape[1]} '
      f'posdir={stack_labels(_f, "cebra_posdir").shape[1]}')

## 3. Light Optuna tuning (centered on the tutorial defaults)

A small Optuna search over a neighbourhood of the rat-hippocampus tutorial defaults
(architecture, learning rate, temperature, time-offset). Trials are scored on the
**valid** split (no test leakage) by the mean over the highest-neuron sessions of the
median 3-D Ridge-decode error, using reduced iterations for speed. The default settings
are enqueued as the first trial, and tuned HPs are adopted **only if they beat the
defaults**. The study is cached to `./out/cebra/`. Set `RUN_CEBRA_OPTUNA=False` to skip
and keep the tutorial defaults.

In [ ]:
import optuna
import pickle

RUN_CEBRA_OPTUNA = True       # False -> skip search, use tutorial defaults
TUNE_VARIANT = 'cebra_pos'    # tune the position-informed model (primary comparison)
TUNE_TRIALS = 20              # light search
TUNE_ITER = 2000              # reduced iters during search (final fit uses CEBRA_MAX_ITER)
TUNE_SESSIONS = sorted(session_names, key=lambda s: unit_num_dict[s], reverse=True)[:3]
optuna_study_path = f'{OUT_DIR}/cebra_optuna_{TUNE_VARIANT}_seed{seed}.pkl'


def _valid_median_err(hp):
    """Mean over TUNE_SESSIONS of the median 3-D Ridge-decode error (m) on VALID."""
    errs = []
    for s_ in TUNE_SESSIONS:
        tr, va = session_flights(s_, 'train'), session_flights(s_, 'valid')
        if not va:
            continue
        set_random_state(seed)
        m = CEBRA(conditional='time_delta', max_iterations=TUNE_ITER, **COMMON_FIXED, **hp)
        m.fit(np.vstack([f['spike'] for f in tr]), stack_labels(tr, TUNE_VARIANT))
        etr, eva = embed_flights(m, tr), embed_flights(m, va)
        Etr = np.vstack([e for e, _ in etr]); Ytr = np.vstack([f['pos'] for _, f in etr])
        Eva = np.vstack([e for e, _ in eva]); Yva = np.vstack([f['pos'] for _, f in eva])
        r = Ridge(alpha=1.0).fit(Etr, Ytr)
        errs.append(float(np.median(np.sqrt(((Yva - r.predict(Eva)) ** 2).sum(1)))))
    return float(np.mean(errs))


def _objective(trial):
    hp = dict(
        model_architecture=trial.suggest_categorical(
            'model_architecture', ['offset5-model', 'offset10-model', 'offset15-model']),
        learning_rate=trial.suggest_float('learning_rate', 1e-4, 3e-3, log=True),
        temperature=trial.suggest_float('temperature', 0.5, 2.0),
        time_offsets=trial.suggest_categorical('time_offsets', [5, 10, 15]),
    )
    return _valid_median_err(hp)


if RUN_CEBRA_OPTUNA:
    if os.path.exists(optuna_study_path):
        cebra_study = pickle.load(open(optuna_study_path, 'rb'))
        print(f'loaded cached study ({len(cebra_study.trials)} trials)')
    else:
        cebra_study = optuna.create_study(
            direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed))
        cebra_study.enqueue_trial(dict(CEBRA_HP))   # tutorial defaults as trial 0
    remaining = max(0, TUNE_TRIALS - len(cebra_study.trials))
    if remaining:
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        cebra_study.optimize(_objective, n_trials=remaining, show_progress_bar=True)
        with open(optuna_study_path, 'wb') as f:
            pickle.dump(cebra_study, f)
    default_err = _valid_median_err(CEBRA_HP)
    print(f'tutorial-default valid err = {default_err:.3f} m')
    print(f'best Optuna  valid err     = {cebra_study.best_value:.3f} m  '
          f'({cebra_study.best_params})')
    if cebra_study.best_value < default_err:
        CEBRA_HP.update(cebra_study.best_params)
        print('-> adopting tuned HPs:', CEBRA_HP)
    else:
        print('-> tuning did not beat defaults; keeping tutorial defaults')
else:
    print('Optuna disabled; using tutorial defaults:', CEBRA_HP)

## 4. Fit CEBRA per session × variant

Each `(session, variant)` is a single-session CEBRA fit on **train** flights only, using
the (possibly tuned) `CEBRA_HP`. Models are cached to `./out/cebra/` under an HP-tagged
name (so tuned and default models never collide). `cebra_time` ignores the label.

In [ ]:
def _hp_tag():
    a = CEBRA_HP['model_architecture'].replace('-model', '')
    return (f"{a}_lr{CEBRA_HP['learning_rate']:.0e}_t{CEBRA_HP['temperature']:.2g}"
            f"_to{CEBRA_HP['time_offsets']}")


def cebra_path(session, variant, seed_):
    return f'{OUT_DIR}/{session}__{variant}__{_hp_tag()}__it{CEBRA_MAX_ITER}__seed{seed_}.pt'


def fit_or_load(session, variant, seed_, max_iter=None):
    max_iter = max_iter or CEBRA_MAX_ITER
    path = cebra_path(session, variant, seed_)
    if os.path.exists(path):
        return CEBRA.load(path)
    train = session_flights(session, 'train')
    X = np.vstack([f['spike'] for f in train])
    set_random_state(seed_)
    model = make_cebra(variant, max_iter)
    if variant == 'cebra_time':
        model.fit(X)
    else:
        model.fit(X, stack_labels(train, variant))
    model.save(path)
    return model


cebra_models = {}   # (session, variant) -> fitted CEBRA (seed=`seed`)
for si, session in enumerate(session_names):
    for variant in VARIANTS:
        cebra_models[(session, variant)] = fit_or_load(session, variant, seed)
    print(f'[{si+1}/{len(session_names)}] {session} fitted ({", ".join(VARIANTS)})')
print('all CEBRA models ready')

## 5. Embed + continuous-linear decode → meters

For each `(session, variant)`: transform train+test flights, fit a **Ridge** read-out
`embedding → xyz (m)` on train, predict test. Records use the exact decode-notebook
per-timestep schema so they slot straight into `all_df`. The CEBRA-tutorial's
`KNeighborsRegressor(metric='cosine')` read-out is also fit and kept in a separate
kNN-vs-linear table.

In [ ]:
def decode_records(session, variant, model):
    """Return (list-of-test-records, knn_row) for one (session, variant)."""
    train = session_flights(session, 'train')
    test = session_flights(session, 'test')
    bat = parse_bat(session)

    emb_tr = embed_flights(model, train)
    emb_te = embed_flights(model, test)
    Etr = np.vstack([e for e, _ in emb_tr]); Ytr = np.vstack([f['pos'] for _, f in emb_tr])
    Ete = np.vstack([e for e, _ in emb_te]); Yte = np.vstack([f['pos'] for _, f in emb_te])

    ridge = Ridge(alpha=1.0).fit(Etr, Ytr)
    # CEBRA-tutorial decoder: pos_decoder = KNeighborsRegressor(n_neighbors, metric='cosine')
    knn = KNeighborsRegressor(n_neighbors=KNN_N, metric='cosine').fit(Etr, Ytr)
    pred_knn = np.asarray(knn.predict(Ete))

    recs = []
    for emb, f in emb_te:
        pred = ridge.predict(emb)        # (T, 3) meters
        true = f['pos']
        e2 = np.sqrt(((true[:, :2] - pred[:, :2]) ** 2).sum(1))
        e3 = np.sqrt(((true - pred) ** 2).sum(1))
        for i in range(len(true)):
            recs.append(dict(bat=bat, session=session, flight_id=f['flight_id'],
                             t_idx=int(f['t_idx'][i]),
                             x_true=float(true[i, 0]), y_true=float(true[i, 1]), z_true=float(true[i, 2]),
                             x_pred=float(pred[i, 0]), y_pred=float(pred[i, 1]), z_pred=float(pred[i, 2]),
                             err_2d=float(e2[i]), err_3d=float(e3[i]), decoder=variant))
    knn_err3d = np.sqrt(((Yte - pred_knn) ** 2).sum(1))
    knn_row = dict(session=session, bat=bat, variant=variant,
                   linear_med_err3d=float(np.median([r['err_3d'] for r in recs])),
                   knn_med_err3d=float(np.median(knn_err3d)))
    return recs, knn_row


cebra_recs = {v: [] for v in VARIANTS}
knn_rows = []
for si, session in enumerate(session_names):
    for variant in VARIANTS:
        recs, krow = decode_records(session, variant, cebra_models[(session, variant)])
        cebra_recs[variant].extend(recs)
        knn_rows.append(krow)
    print(f'[{si+1}/{len(session_names)}] {session} decoded')

cebra_dfs = {v: pd.DataFrame(cebra_recs[v]) for v in VARIANTS}
knn_df = pd.DataFrame(knn_rows)
print('per-variant pooled-median test 3D err (m):')
for v in VARIANTS:
    print(f'  {v:14s} {cebra_dfs[v]["err_3d"].median():.3f}')

In [ ]:
# Linear vs kNN read-out (sanity / reference table).
knn_summary = (knn_df.groupby('variant')[['linear_med_err3d', 'knn_med_err3d']]
               .median().reset_index())
print('Median-over-sessions 3D err (m), linear (Ridge) vs kNN read-out:')
knn_summary

## 6. Transformer reference (same DEFAULT split)

Rebuild the best Transformer decoder from the saved Optuna study's `best_params`
(no re-search) and train it on the `DEFAULT_SESSION_NAMES` split (cached). Because the
per-session readin/readout `ModuleDict`s are keyed by session, the checkpoint trained on
the full session list won't load here — so we retrain once with the fitted HPs. Then
produce `transformer_df` with the same per-timestep schema via the decode-notebook eval.

In [ ]:
import pickle

fixed_config = dict(input_dim_dict=unit_num_dict, output_dim=3, num_heads=4,
                    final_activation=nn.Identity())

def make_decoder(params):
    return Bat_Transformer_Decoder(
        latent_dim=params['latent_dim'], ff_dim=params['ff_dim'],
        num_layers=params['num_layers'], dropout_prob=params['dropout_prob'],
        readin_dropout=params['readin_dropout'], input_transform=params['input_transform'],
        **fixed_config)

def _decoder_loss(name):
    return {'mse': nn.MSELoss(), 'huber': nn.SmoothL1Loss(), 'l1': nn.L1Loss()}[name]

def _neg_euclid_z(y_true, y_pred):
    return -np.sqrt(((y_true - y_pred) ** 2).sum(axis=1))

def _substr(p):
    return '_'.join(f'{k}_{v:.4g}' if isinstance(v, float) else f'{k}_{v}'
                    for k, v in p.items())

study_path = f'./out/bat_decoder/bat_decoder_seed_{seed}_optuna_metersearch.pkl'
best_params = pickle.load(open(study_path, 'rb')).best_params
print('best_params:', best_params)

# Cache under a DEFAULT-split-specific name so it never collides with the
# all-session checkpoints written by decode-position-comparison.ipynb.
tform_path = f'./out/bat_decoder/cebra_cmp_DEFAULT_seed{seed}_{_substr(best_params)}.pth'

set_random_state(seed)
tform = make_decoder(best_params)
if os.path.exists(tform_path):
    tform.load_state_dict(torch.load(tform_path, map_location=device))
    tform = tform.to(device)
    print('loaded', tform_path)
else:
    bat_decoder_trainer(model=tform, dataloaders=dataloaders, device=device, pos_dims=3,
                        n_epochs=1000, lr=best_params['lr'],
                        weight_decay=best_params['weight_decay'],
                        loss_fn=_decoder_loss(best_params['loss_name']),
                        valid_fn=_neg_euclid_z, valid_stat='median',
                        early_stopping_patience=50, save_model=True, model_name=tform_path)
    print('trained ->', tform_path)

In [ ]:
def transformer_records(model):
    recs = []
    model.eval()
    with torch.no_grad():
        for session, sd in dataloaders['test'].items():
            bm = meta_info_dict['behavior_mean'][session].squeeze()
            bs = meta_info_dict['behavior_std'][session].squeeze()
            bat = parse_bat(session)
            for batch in sd:
                X = batch.spike.to(device); m = batch.mask.to(device)
                yt_z = batch.behavior[:, :, :3].numpy()
                yp_z = model(session, X, m).cpu().numpy()
                mask = batch.mask.numpy().astype(bool); B, T = mask.shape
                yt = yt_z * bs[None, None, :3] + bm[None, None, :3]
                yp = yp_z * bs[None, None, :3] + bm[None, None, :3]
                fpi = np.broadcast_to(batch.flight_path_index.numpy()[:, None], (B, T))
                tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
                ytf, ypf = yt[mask], yp[mask]
                e2 = np.sqrt(((ytf[:, :2] - ypf[:, :2]) ** 2).sum(1))
                e3 = np.sqrt(((ytf - ypf) ** 2).sum(1))
                fid, tid = fpi[mask].astype(int), tix[mask].astype(int)
                for i in range(len(e2)):
                    recs.append(dict(bat=bat, session=session, flight_id=int(fid[i]),
                                     t_idx=int(tid[i]),
                                     x_true=float(ytf[i, 0]), y_true=float(ytf[i, 1]), z_true=float(ytf[i, 2]),
                                     x_pred=float(ypf[i, 0]), y_pred=float(ypf[i, 1]), z_pred=float(ypf[i, 2]),
                                     err_2d=float(e2[i]), err_3d=float(e3[i]), decoder='transformer'))
    return pd.DataFrame(recs)

transformer_df = transformer_records(tform)
print('transformer pooled-median test 3D err (m):', transformer_df['err_3d'].median())

## 7. Metric A — decoding comparison

Combine all decoders into one long `all_df`, compute per-session medians and per-flight
discrete-Fréchet distances, and compare with per-session paired Wilcoxon tests
(session-level, nonparametric, 3-tier stars, n.s. hidden), colored by bat.

In [ ]:
DECODER_ORDER = ['cebra_time', 'cebra_pos', 'cebra_posdir', 'transformer']
DECODER_LABEL = {'cebra_time': 'CEBRA-Time', 'cebra_pos': 'CEBRA-Pos',
                 'cebra_posdir': 'CEBRA-PosDir', 'transformer': 'Transformer'}

all_df = pd.concat([cebra_dfs['cebra_time'], cebra_dfs['cebra_pos'],
                    cebra_dfs['cebra_posdir'], transformer_df], ignore_index=True)
all_df['decoder'] = pd.Categorical(all_df['decoder'], categories=DECODER_ORDER, ordered=True)

median_per_session = (all_df.groupby(['bat', 'session', 'decoder'], observed=True)
                      [['err_2d', 'err_3d']].median().reset_index())


def compute_frechet_per_flight(df):
    rows = []
    for (bat, session, fid, dec), grp in df.groupby(
            ['bat', 'session', 'flight_id', 'decoder'], observed=True, sort=False):
        grp = grp.sort_values('t_idx')
        truth = grp[['x_true', 'y_true', 'z_true']].to_numpy(np.float64)
        pred = grp[['x_pred', 'y_pred', 'z_pred']].to_numpy(np.float64)
        rows.append(dict(bat=bat, session=session, flight_id=int(fid),
                         decoder=dec, n_t=len(grp),
                         frechet_2d=discrete_frechet(truth[:, :2], pred[:, :2]),
                         frechet_3d=discrete_frechet(truth, pred)))
    out = pd.DataFrame(rows)
    out['decoder'] = pd.Categorical(out['decoder'], categories=DECODER_ORDER, ordered=True)
    return out

frechet_df = compute_frechet_per_flight(all_df)
frechet_med = (frechet_df.groupby(['bat', 'session', 'decoder'], observed=True)
               [['frechet_2d', 'frechet_3d']].median().reset_index())
median_per_session = median_per_session.merge(frechet_med, on=['bat', 'session', 'decoder'], how='left')
median_per_session.head()

In [ ]:
from itertools import combinations
from scipy.stats import wilcoxon


def _stars(p):
    if not np.isfinite(p):
        return None
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else None


def _bracket(ax, x1, x2, y, p, h_frac=0.03):
    s = _stars(p)
    if s is None:          # hide n.s.
        return
    h = h_frac * (ax.get_ylim()[1] - ax.get_ylim()[0])
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='k', lw=0.9)
    ax.text((x1 + x2) / 2, y + h, s, ha='center', va='bottom', fontsize=11)


def pairwise_across_session(median_df, col):
    wide = median_df.pivot(index='session', columns='decoder', values=col).dropna()
    out = {}
    for d1, d2 in combinations(DECODER_ORDER, 2):
        if d1 not in wide or d2 not in wide:
            continue
        diff = (wide[d2] - wide[d1]).values
        if np.all(diff == 0):
            out[(d1, d2)] = (np.nan, len(diff), float(np.median(diff)))
        else:
            r = wilcoxon(diff, zero_method='wilcox', alternative='two-sided')
            out[(d1, d2)] = (float(r.pvalue), len(diff), float(np.median(diff)))
    return out


METRIC_PANELS = [('err_2d', 'Per-session median 2D error', 'Median error (m)'),
                 ('err_3d', 'Per-session median 3D error', 'Median error (m)'),
                 ('frechet_2d', 'Per-session median 2D Fréchet', 'Median Fréchet (m)'),
                 ('frechet_3d', 'Per-session median 3D Fréchet', 'Median Fréchet (m)')]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()
xpos = {d: i for i, d in enumerate(DECODER_ORDER)}
for ax, (col, title, ylab) in zip(axes, METRIC_PANELS):
    sns.lineplot(data=median_per_session, x='decoder', y=col, hue='bat',
                 palette=BAT_PALETTE, units='session', estimator=None, marker='o',
                 lw=1.2, alpha=0.85, ax=ax, legend=(ax is axes[0]))
    ax.set_title(title); ax.set_ylabel(ylab); ax.set_xlabel('Model')
    ax.set_xticks(range(len(DECODER_ORDER)))
    ax.set_xticklabels([DECODER_LABEL[d] for d in DECODER_ORDER], rotation=15, ha='right')
    ax.set_ylim(bottom=0)
    if ax is axes[0]:
        ax.legend(title='bat', bbox_to_anchor=(1.02, 1.0), loc='upper left',
                  fontsize=7, frameon=False, ncol=2)
    # adjacent + endpoint brackets
    pairs = pairwise_across_session(median_per_session, col)
    ytop = ax.get_ylim()[1]
    levels = {('cebra_time', 'cebra_pos'): 0.86, ('cebra_pos', 'cebra_posdir'): 0.86,
              ('cebra_posdir', 'transformer'): 0.86, ('cebra_time', 'transformer'): 0.97}
    for (d1, d2), frac in levels.items():
        if (d1, d2) in pairs:
            _bracket(ax, xpos[d1], xpos[d2], ytop * frac, pairs[(d1, d2)][0])
    ax.set_ylim(top=ytop * 1.15)
plt.tight_layout(); plt.show()

print('Pooled median test error (m):')
print(all_df.groupby('decoder', observed=True)[['err_2d', 'err_3d']].median())
for col, lab in [('err_3d', '3D err'), ('frechet_3d', '3D Fréchet')]:
    print(f'\nAcross-session paired Wilcoxon ({lab}):')
    for (d1, d2), (p, n, md_) in pairwise_across_session(median_per_session, col).items():
        print(f'  {DECODER_LABEL[d1]:>12} vs {DECODER_LABEL[d2]:<12} '
              f'n={n:2d} median(diff)={md_:+.3f} m  p={p:.3g}')

## 8. Metric B — `consistency_score` between runs

Refit each `(session, variant)` for `N_SEEDS` seeds and measure CEBRA's between-runs
consistency (R² after linear alignment) — the standard embedding-reproducibility check
from the CEBRA demos.

In [ ]:
from cebra.integrations.sklearn.metrics import consistency_score

runs_rows = []
for si, session in enumerate(session_names):
    train = session_flights(session, 'train')
    X = np.vstack([f['spike'] for f in train])
    for variant in ['cebra_pos', 'cebra_posdir']:
        embs = []
        for s_ in range(N_SEEDS):
            m = fit_or_load(session, variant, s_)         # seed 0..N-1 (cached)
            embs.append(m.transform(X))
        scores, _, _ = consistency_score(embeddings=embs, between='runs')
        runs_rows.append(dict(session=session, bat=parse_bat(session), variant=variant,
                              mean_r2=float(np.mean(scores)), min_r2=float(np.min(scores))))
    print(f'[{si+1}/{len(session_names)}] {session} between-runs done')

runs_df = pd.DataFrame(runs_rows)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=runs_df, x='variant', y='mean_r2', color='lightgray', width=0.5, ax=ax)
sns.stripplot(data=runs_df, x='variant', y='mean_r2', hue='bat', palette=BAT_PALETTE,
              size=6, ax=ax)
ax.set_ylabel('Between-runs consistency R²'); ax.set_ylim(0, 1)
ax.legend(title='bat', bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=7, frameon=False, ncol=2)
ax.set_title(f'CEBRA embedding reproducibility ({N_SEEDS} seeds/session)')
plt.tight_layout(); plt.show()
runs_df.groupby('variant')[['mean_r2', 'min_r2']].median()

## 9. Metric C — `consistency_score` across sessions / bats

Compare embeddings across sessions, aligned on **per-arena-normalized** position
(scale each session's `x, y` to `[0,1]` via `meta_info_dict['r_lim']`; `z` by its
observed range). With only `DEFAULT_SESSION_NAMES` (all env `'lab'`), we group the
resulting pairwise R² into **same-bat** vs **cross-bat**.

In [ ]:
from scipy.stats import rankdata

# consistency_score(between='datasets') aligns on a *1D* label, digitizing it
# into bins that must each be populated.  A discrete 3D-voxel index leaves empty
# bins and crashes the aligner, so we align on each spatial axis separately,
# per-session quantile-normalized to a uniform [0,1] (every bin populated by
# construction), and average the resulting R² across the three axes.
N_ALIGN_BINS = 50

def quantile_uniform(v):
    """Map values to their uniform [0,1] rank (dense, gap-free coverage)."""
    return (rankdata(v, method='average') - 0.5) / len(v)


def cross_dataset_consistency(variant):
    # Build each session's embedding + per-axis quantile labels once.
    embs, ids, axis_labels = [], [], []
    for session in session_names:
        flights = session_flights(session, 'train') + session_flights(session, 'test')
        X = np.vstack([f['spike'] for f in flights])
        pos = np.vstack([f['pos'] for f in flights])
        embs.append(cebra_models[(session, variant)].transform(X))
        axis_labels.append([quantile_uniform(pos[:, a]) for a in range(3)])
        ids.append(session)

    # Average per-pair R² across the three spatial axes; then average the two
    # ordered directions consistency_score returns into one unordered-pair R².
    acc = {}
    for a in range(3):
        labels_a = [al[a] for al in axis_labels]
        scores, pairs, _ = consistency_score(
            embeddings=embs, labels=labels_a, dataset_ids=ids,
            between='datasets', num_discretization_bins=N_ALIGN_BINS)
        for pr, sc in zip(pairs, np.asarray(scores).ravel()):
            acc.setdefault(frozenset((str(pr[0]), str(pr[1]))), []).append(float(sc))

    rows = []
    for key, vals in acc.items():
        sa, sb = sorted(key)
        rows.append(dict(variant=variant, session_a=sa, session_b=sb, r2=float(np.mean(vals)),
                         group='same-bat' if parse_bat(sa) == parse_bat(sb) else 'cross-bat'))
    return pd.DataFrame(rows)

cross_df = pd.concat([cross_dataset_consistency(v) for v in ['cebra_pos', 'cebra_posdir']],
                     ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=cross_df, x='variant', y='r2', hue='group', width=0.6, ax=ax)
ax.set_ylabel('Between-datasets consistency R²'); ax.set_ylim(0, 1)
ax.set_title('Cross-session CEBRA consistency (position-aligned)')
plt.tight_layout(); plt.show()

print(cross_df.groupby(['variant', 'group'])['r2'].agg(['median', 'count']))
for v in ['cebra_pos', 'cebra_posdir']:
    sub = cross_df[cross_df.variant == v]
    sb = sub[sub.group == 'same-bat']['r2']; cb = sub[sub.group == 'cross-bat']['r2']
    if len(sb) and len(cb):
        from scipy.stats import mannwhitneyu
        p = mannwhitneyu(sb, cb, alternative='greater').pvalue
        print(f'{v}: same-bat median {sb.median():.3f} > cross-bat {cb.median():.3f}? p={p:.3g}')

## 10. Persistent cohomology (topological data analysis)

Following Schneider et al. (2023, Fig. 2e–g), we probe the *shape* of the CEBRA
embedding with persistent cohomology (`ripser`). Because the models use a cosine
objective the embeddings live on the unit hypersphere, so we compute persistence
diagrams of H0 (components), H1 (loops) and H2 (voids) directly. Betti numbers are read
off by counting features whose lifespan exceeds a **shuffled-embedding null** (per-dim
max lifespan over column-shuffled draws, which destroys topological structure); the
single infinite H0 bar is always counted as the one connected component.

Unlike the rat 1.6 m linear track (a ring → Betti (1,1,0)), bat flight fills an open
3-D volume, so this is exploratory — we contrast CEBRA-Pos, CEBRA-Time and a shuffled
control. `ripser` at maxdim 2 is cubic in sample count, so we subsample and use the
highest-neuron sessions; tune `TDA_*` to trade speed for resolution.

In [ ]:
from ripser import ripser

TDA_MAXDIM = 2     # H0 components, H1 loops, H2 voids
TDA_COEFF = 2      # Z/2 coefficients
TDA_SUB = 600      # subsample (ripser maxdim=2 is ~cubic in n)
TDA_SHUFFLE = 2    # column-shuffle null draws
TDA_VARIANTS = ['cebra_pos', 'cebra_time']
TDA_SESSIONS = sorted(session_names, key=lambda s: unit_num_dict[s], reverse=True)[:3]
_tda_rng = np.random.default_rng(seed)


def embed_pooled(session, variant):
    flights = session_flights(session, 'train') + session_flights(session, 'test')
    X = np.vstack([f['spike'] for f in flights])
    return cebra_models[(session, variant)].transform(X)


def _sub(E):
    if len(E) <= TDA_SUB:
        return E
    return E[_tda_rng.choice(len(E), TDA_SUB, replace=False)]


def _shuffle_cols(E):
    return np.column_stack([_tda_rng.permutation(E[:, j]) for j in range(E.shape[1])])


def betti_and_dgms(E, shuffle_input=False):
    """Persistence diagrams + Betti numbers vs a column-shuffled null.
    shuffle_input=True first scrambles the embedding itself (topology control)."""
    E = _sub(E)
    if shuffle_input:
        E = _shuffle_cols(E)
    real = ripser(E, maxdim=TDA_MAXDIM, coeff=TDA_COEFF)['dgms']
    thr = [0.0] * (TDA_MAXDIM + 1)
    for _ in range(TDA_SHUFFLE):
        sh = ripser(_shuffle_cols(E), maxdim=TDA_MAXDIM, coeff=TDA_COEFF)['dgms']
        for d, dg in enumerate(sh):
            ls = dg[:, 1] - dg[:, 0]; ls = ls[np.isfinite(ls)]
            if len(ls):
                thr[d] = max(thr[d], float(ls.max()))
    betti = []
    for d, dg in enumerate(real):
        ls = dg[:, 1] - dg[:, 0]
        n_inf = int((~np.isfinite(ls)).sum())
        betti.append(n_inf + int((ls[np.isfinite(ls)] > thr[d]).sum()))
    return betti, real, thr


tda_rows, tda_dgms = [], {}
for session in TDA_SESSIONS:
    for variant in TDA_VARIANTS:
        bn, dg, thr = betti_and_dgms(embed_pooled(session, variant))
        tda_dgms[(session, variant)] = (dg, thr, bn)
        tda_rows.append(dict(session=session, bat=parse_bat(session), variant=variant,
                             H0=bn[0], H1=bn[1], H2=bn[2] if len(bn) > 2 else 0))
    print(f'{session} TDA done')

# Shuffled-embedding control on the representative (top-neuron) session.
rep = TDA_SESSIONS[0]
bn_sh, dg_sh, thr_sh = betti_and_dgms(embed_pooled(rep, 'cebra_pos'), shuffle_input=True)
tda_dgms[(rep, 'shuffled')] = (dg_sh, thr_sh, bn_sh)
tda_rows.append(dict(session=rep, bat=parse_bat(rep), variant='shuffled',
                     H0=bn_sh[0], H1=bn_sh[1], H2=bn_sh[2] if len(bn_sh) > 2 else 0))

tda_df = pd.DataFrame(tda_rows)
print('\nBetti numbers (features beating the shuffled-embedding null):')
print(tda_df.to_string(index=False))

In [ ]:
# Persistence diagrams for the representative session: CEBRA-Pos, CEBRA-Time, shuffled.
DIM_COLOR = {0: 'tab:gray', 1: 'tab:red', 2: 'tab:blue'}
DIM_NAME = {0: 'H0', 1: 'H1 loops', 2: 'H2 voids'}
panels = [(rep, 'cebra_pos'), (rep, 'cebra_time'), (rep, 'shuffled')]
TITLE = {'cebra_pos': 'CEBRA-Pos', 'cebra_time': 'CEBRA-Time', 'shuffled': 'Shuffled'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, key in zip(axes, panels):
    dgms, thr, bn = tda_dgms[key]
    finite = [dg[np.isfinite(dg[:, 1])] for dg in dgms]
    hi = max([f[:, 1].max() for f in finite if len(f)] + [1.0]) * 1.05
    ax.plot([0, hi], [0, hi], 'k--', lw=0.8, zorder=0)
    for d, dg in enumerate(dgms):
        pts = dg[np.isfinite(dg[:, 1])]
        if len(pts):
            ax.scatter(pts[:, 0], pts[:, 1], s=18, color=DIM_COLOR[d], alpha=0.8,
                       label=f'{DIM_NAME[d]} (β={bn[d]})')
        if thr[d] > 0:   # null lifespan band: death = birth + thr
            ax.plot([0, hi], [thr[d], hi + thr[d]], color=DIM_COLOR[d], lw=0.6, ls=':')
    ax.set_xlabel('Birth'); ax.set_ylabel('Death'); ax.set_xlim(0); ax.set_ylim(0)
    ax.set_title(f"{TITLE[key[1]]}  —  Betti {tuple(bn)}")
    ax.legend(fontsize=8, frameon=False, loc='lower right')
fig.suptitle(f'Persistence diagrams (dotted = shuffled-null lifespan threshold)  |  {rep}')
plt.tight_layout(); plt.show()

## 11. Multi-session vs single-session: consistency & position decoding

Following Schneider et al. (2023, Ext. Data Fig. 7a–c, **position-decoding panel only**),
we jointly train **one** CEBRA model across all sessions — one encoder per session into a
shared, session-invariant embedding space (`fit([X_1,…], [y_1,…])`, position label) — and
compare it to the independent single-session models from Section 4. We report, with the
paper's metric (R² of linear regression between aligned embeddings):

1. **Consistency** of every session pair, single vs multi (Fig. 7a/b) — multi-session
   training is expected to *raise* cross-session consistency.
2. **Position decoding error vs consistency** (Fig. 7c) — single (circles) vs multi
   (triangles); the paper finds joint training raises consistency at *no cost* to
   decoding.

In [ ]:
# --- One multi-session CEBRA over all sessions (position label) ---
MULTI_VARIANT = 'cebra_pos'
multi_path = (f'{OUT_DIR}/multisession_{MULTI_VARIANT}__{_hp_tag()}'
              f'__it{CEBRA_MAX_ITER}__seed{seed}.pt')

_multi_train = {s: session_flights(s, 'train') for s in session_names}
Xs = [np.vstack([f['spike'] for f in _multi_train[s]]) for s in session_names]
Ys = [stack_labels(_multi_train[s], MULTI_VARIANT) for s in session_names]

if os.path.exists(multi_path):
    multi_model = CEBRA.load(multi_path)
    print(f'loaded multi-session model ({multi_model.num_sessions_} sessions)')
else:
    set_random_state(seed)
    multi_model = CEBRA(conditional='time_delta', max_iterations=CEBRA_MAX_ITER,
                        **COMMON_FIXED, **CEBRA_HP)
    multi_model.fit(Xs, Ys)     # multi-session: list of X (varied neuron counts), list of y
    multi_model.save(multi_path)
    print(f'trained multi-session model ({multi_model.num_sessions_} sessions)')


def embed_flights_multi(model, flights, session_id):
    """Per-flight transform from a multi-session model (needs session_id)."""
    req = len(model.offset_)
    out = []
    for f in flights:
        Xp, n = _pad(f['spike'], req)
        out.append((model.transform(Xp, session_id=session_id)[:n], f))
    return out

In [ ]:
from scipy.stats import rankdata, wilcoxon


def pairwise_consistency(emb_list, pos_list, ids):
    """Paper consistency: between-datasets linear-regression R², aligned per spatial
    axis on a per-session quantile-uniform label (Section 8 alignment), averaged over
    x/y/z and both directions.  Returns one row per unordered session pair."""
    axis_labels = [[(rankdata(p[:, a], method='average') - 0.5) / len(p) for a in range(3)]
                   for p in pos_list]
    acc = {}
    for a in range(3):
        scores, pairs, _ = consistency_score(
            embeddings=emb_list, labels=[al[a] for al in axis_labels],
            dataset_ids=ids, between='datasets', num_discretization_bins=50)
        for pr, sc in zip(pairs, np.asarray(scores).ravel()):
            acc.setdefault(frozenset((str(pr[0]), str(pr[1]))), []).append(float(sc))
    rows = []
    for key, vals in acc.items():
        sa, sb = sorted(key)
        rows.append(dict(session_a=sa, session_b=sb, r2=float(np.mean(vals)),
                         same_bat=parse_bat(sa) == parse_bat(sb)))
    return pd.DataFrame(rows)


# Build single + multi embeddings (train+test pooled) and per-session decoding error.
single_emb, multi_emb, pos_pooled = [], [], []
dec_rows = []
for i, s in enumerate(session_names):
    tr, te = _multi_train[s], session_flights(s, 'test')
    s_tr = embed_flights(cebra_models[(s, MULTI_VARIANT)], tr)
    s_te = embed_flights(cebra_models[(s, MULTI_VARIANT)], te)
    m_tr = embed_flights_multi(multi_model, tr, i)
    m_te = embed_flights_multi(multi_model, te, i)
    for tag, etr, ete in [('single', s_tr, s_te), ('multi', m_tr, m_te)]:
        Etr = np.vstack([e for e, _ in etr]); Ytr = np.vstack([f['pos'] for _, f in etr])
        Ete = np.vstack([e for e, _ in ete]); Yte = np.vstack([f['pos'] for _, f in ete])
        err_cm = float(np.median(np.sqrt(((Yte - Ridge(alpha=1.0).fit(Etr, Ytr).predict(Ete))
                                          ** 2).sum(1)))) * 100
        dec_rows.append(dict(session=s, bat=parse_bat(s), training=tag, err_cm=err_cm))
    single_emb.append(np.vstack([e for e, _ in s_tr + s_te]))
    multi_emb.append(np.vstack([e for e, _ in m_tr + m_te]))
    pos_pooled.append(np.vstack([f['pos'] for f in tr + te]))

con_single = pairwise_consistency(single_emb, pos_pooled, session_names).assign(training='single')
con_multi = pairwise_consistency(multi_emb, pos_pooled, session_names).assign(training='multi')
dec_df = pd.DataFrame(dec_rows)

# Fig 7a: does joint training raise cross-session consistency? (paired, one-sided)
merged = con_single.merge(con_multi, on=['session_a', 'session_b'], suffixes=('_single', '_multi'))
w = wilcoxon(merged['r2_multi'], merged['r2_single'], alternative='greater')
print(f'consistency: single median {con_single.r2.median():.3f}  ->  '
      f'multi median {con_multi.r2.median():.3f}  '
      f'(paired Wilcoxon multi>single p={w.pvalue:.3g}, n={len(merged)})')
dpv = wilcoxon(dec_df[dec_df.training == 'multi'].set_index('session').err_cm,
               dec_df[dec_df.training == 'single'].set_index('session').err_cm
               ).pvalue
print(f'decoding err (cm): single median '
      f'{dec_df[dec_df.training=="single"].err_cm.median():.1f}  ->  '
      f'multi median {dec_df[dec_df.training=="multi"].err_cm.median():.1f}  '
      f'(paired Wilcoxon two-sided p={dpv:.3g})')

In [ ]:
# Fig 7a/b style: paired consistency single -> multi, and the consistency matrices.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.4]})

# (left) paired lines per session-pair
ax = axes[0]
for _, r in merged.iterrows():
    ax.plot([0, 1], [r['r2_single'] * 100, r['r2_multi'] * 100],
            color='0.6', lw=0.8, alpha=0.6, marker='o', ms=3)
ax.set_xticks([0, 1]); ax.set_xticklabels(['single', 'multi'])
ax.set_ylabel('Cross-session consistency R² [%]')
ax.set_title(f'Joint training raises consistency\n(Wilcoxon p={w.pvalue:.2g}, n={len(merged)})')
ax.set_xlim(-0.3, 1.3)

# (right) decoding error vs consistency (Fig 7c, position panel)
ax = axes[1]
con_by = {'single': con_single, 'multi': con_multi}
scatter = []
for training in ('single', 'multi'):
    cm = con_by[training]
    for s in session_names:
        sub = cm[(cm.session_a == s) | (cm.session_b == s)]
        err = dec_df[(dec_df.session == s) & (dec_df.training == training)].err_cm.iloc[0]
        scatter.append(dict(session=s, bat=parse_bat(s), training=training,
                            mean_r2_pct=float(sub.r2.mean()) * 100, err_cm=err))
scatter_df = pd.DataFrame(scatter)
for training, mk in [('single', 'o'), ('multi', 'v')]:
    d = scatter_df[scatter_df.training == training]
    sns.scatterplot(data=d, x='mean_r2_pct', y='err_cm', hue='bat', palette=BAT_PALETTE,
                    marker=mk, s=70, ax=ax, legend=(training == 'single'), edgecolor='k')
ax.invert_yaxis()   # paper plots lower error at the top
ax.set_xlabel('Consistency, R² [%]'); ax.set_ylabel('Decoding error [cm]')
ax.set_title('Position decoding vs consistency\n(○ single, ▽ multi)')
ax.legend(title='bat', bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=7,
          frameon=False, ncol=2)
plt.tight_layout(); plt.show()

print(scatter_df.groupby('training')[['mean_r2_pct', 'err_cm']].median())

## Summary

- **Metric A** quantifies how much position a *linear* read-out can recover from each
  embedding, in directly comparable meters/Fréchet units, against the supervised
  Transformer (expected to be the strongest). A kNN (`KNeighborsRegressor`, cosine)
  read-out is reported alongside.
- **Metric B** confirms CEBRA embeddings are reproducible across seeds.
- **Metric C** asks whether the position-aligned latent geometry is more consistent
  within a bat than across bats.
- **Section 10** reads the topology of the embedding (persistent cohomology / Betti
  numbers) vs a shuffled null, after Schneider et al. (2023).
- **Section 11** reproduces the paper's multi- vs single-session consistency analysis
  (Ext. Data Fig. 7, position decoding): a jointly-trained model raises cross-session
  consistency, ideally at no cost to decoding.

Deferred (see plan): Krishna sessions + true cross-environment consistency.